In [1]:
import os, sys, importlib
from google.colab import drive
drive.mount('/content/Drive')

sys.path.append(os.path.abspath("/content/Drive/MyDrive/airbnbMM2/host_about"))
print(sys.path)
from utils import zsc

Mounted at /content/Drive
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/Drive/MyDrive/airbnbMM2/host_about']


## input

In [11]:
DATA_FOLDER="/content/Drive/MyDrive/airbnbMM2/host_about/data_processed"
IMAGES_FOLDER="/content/drive/MyDrive/airbnbMM2/host_pic/images"

os.makedirs(IMAGES_FOLDER, exist_ok=True)

input_output={}
for i, folder in enumerate([f for f in os.listdir(DATA_FOLDER) if not f.endswith('.csv')]):
    print(i, folder)

    # folder = city_time

    # ---listings---
    files=os.listdir(os.path.join(DATA_FOLDER, folder))
    for f in files:
        if f.startswith('listings_processed'):
            path_listings=os.path.join(DATA_FOLDER, folder, f)
            path_listings_zsc=os.path.join(DATA_FOLDER, folder, f"listings_zsc_{folder}.csv")
    path_results_zsc=os.path.join(DATA_FOLDER, folder, f"resultsZSC_{folder}.csv")

    # ---images folder---
    pic_folder=os.path.join(IMAGES_FOLDER, folder)
    os.makedirs(pic_folder, exist_ok=True)

    # --resultsface---
    resultsFACE_folder=os.path.join(IMAGES_FOLDER, 'results')
    os.makedirs(resultsFACE_folder, exist_ok=True)

    path_results_face=os.path.join(resultsFACE_folder, f"face_{folder}.json")
    path_results_deepface=os.path.join(resultsFACE_folder, f"deepface_{folder}.json")

    # ---gather---
    input_output[folder]={"listings":path_listings,
              "results_zsc":path_results_zsc,
              "listings_zsc":path_listings_zsc,

              'pic_folder':pic_folder,
              'results_face':path_results_face,
              "results_deepface":path_results_deepface}

for k,v in input_output.items():
    print(f"{k}".center(100,'-'))
    path_listings=v['listings']
    path_results_zsc=v['results_zsc']
    path_listings_zsc=v['listings_zsc']

    pic_folder=v['pic_folder']
    path_results_face=v['results_face']
    path_results_deepface=v['results_deepface']

    print(path_listings)
    print(path_results_zsc)
    print(path_listings_zsc,"\n")
    print("pic folder:",pic_folder)
    print(path_results_face)
    print(path_results_deepface,"\n")

0 paris_2306
1 london_2406
2 london_2306
3 paris_2406
4 paris_2312
5 london_2312
---------------------------------------------paris_2306---------------------------------------------
/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/listings_processed_paris_2306_32901.csv
/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/resultsZSC_paris_2306.csv
/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2306/listings_zsc_paris_2306.csv 

pic folder: /content/drive/MyDrive/airbnbMM2/host_pic/images/paris_2306
/content/drive/MyDrive/airbnbMM2/host_pic/images/results/face_paris_2306.json
/content/drive/MyDrive/airbnbMM2/host_pic/images/results/deepface_paris_2306.json 

--------------------------------------------london_2406---------------------------------------------
/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/london_2406/listings_processed_london_2406_49856.csv
/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/london

## zsc

In [ ]:
import pandas as pd
import numpy as np
import time, os, sys,re
from tqdm import tqdm
from datetime import datetime
from transformers import pipeline
import torch

"""

classifier_bge = pipeline("zero-shot-classification", model="MoritzLaurer/bge-m3-zeroshot-v2.0", use_safetensors=False,device=2)

sequences = [
    "The government passed a new policy.",
    "This dish uses a lot of spices.",
    "She won the tennis championship."
]

labels = ["politics", "cooking", "sports"]

results = classifier_bge(sequences, candidate_labels=labels, multi_label=True)


"""


def zsc_text(row, classifier, by_lang, dict_items):
        # host_about
        text = row["host_about"]

        # labels
        if by_lang :
            # zsc par langue ; si pas en fr, utilise les labels anglais
            # 若分语言，寻找对应语言的items，若没有对应语言的items，默认en items
            lang= row["lang"]
            candidate_labels=dict_items.get(lang, dict_items['en'])
        else :
            candidate_labels=dict_items['en']

        # zsc
        dict_fr2en=dict(zip(dict_items["fr"], dict_items["en"]))

        try:
            res = classifier(text, candidate_labels, multi_label=True)#***
            # labels fr2en:
            if by_lang and lang == "fr":
                labels_en = [dict_fr2en[label] for label in res["labels"]]
            else:
                labels_en = res["labels"]  # already EN
            dict_scores = dict(zip(labels_en, res["scores"]))

            return  {
                    "host_id": row["host_id"],
                    "host_about":text,
                    # "lang": lang,
                    **dict_scores
                }

        except Exception as e:
            # 报错则将所有labels en初始化为nan
            all_en_labels = dict_fr2en.values() if lang == "fr" else dict_items["en"]
            return {
                "host_id": row["host_id"],
                "host_about":text,
                # "lang": lang,
                **{label: np.nan for label in all_en_labels}
            }



def run_zsc(
    df_input,
    model_name="MoritzLaurer/bge-m3-zeroshot-v2.0", #"MoritzLaurer/bge-m3-zeroshot-v2.0", #"tasksource/ModernBERT-large-nli",#par défaut
    by_lang=True,
    path_results=None,
    save_interval=1000,
    ):

    """
    INPUT : df_processed

    OUTPUT :
        results: csv :
            host_id:int, host_about:str, items:int
        df_zsced=df_processed.merge(results_zsc)

    """

    # ---input: df_unique---
    df=df_input.copy()
    df_unique=df.dropna(subset="host_about").drop_duplicates(subset="host_about")
    print(f"[info] df: {len(df)}; df_unique : {len(df_unique)}\n")


    # ---dict_items---
    labels_en=[
        'open to different cultures', 'cosmopolitan','international view', 'cultural exchange',
        'personal life', 'life experiences', 'divers interests', 'hobbies', 'enjoy life',
        'meet new people', 'welcoming', 'friendly', 'sociable', 'interpersonal interaction',
        'thoughtful service', 'attentive to needs', 'willing to help', 'responsive',
        'fan of Airbnb', 'Airbnb community','love Airbnb', 'travel with Airbnb'
    ]
    labels_fr=[
        'ouvert aux différentes cultures', 'cosmopolite','vue internationale', 'échange culturel',
        'vie personnelle', 'parcours personnel', 'loisirs', 'passions', 'aimer la vie',
        'rencontrer de nouvelles personnes', 'accueillant', 'amical','sociable', 'interaction interpersonnelle',
        'rendre service','attentif aux besoins', 'prêt à aider', 'réactif',
        "adepte d'Airbnb",'communauté Airbnb', 'aime Airbnb', 'voyager par Airbnb'
        ]
    dict_items={'en':labels_en,
                'fr':labels_fr}
    # dict_fr2en=dict(zip(dict_items["fr"], dict_items["en"]))

    if len(labels_en)!=len(labels_fr):
        print(f"[CHECK] labels fr match labels en !")

    # ---IO:no repetition---
    # NB. host_id, id都是唯一的，但是host_about不是！

    if os.path.exists(path_results):
        results_df=pd.read_csv(path_results)
        processed=set(results_df['host_about'])
        pending_df=df_unique[~df_unique['host_about'].isin(processed)]

        print(f"[info] {len(processed)} rows already zsced; {len(pending_df)} rows to zsc!\n")

    else :
        pending_df=df_unique.copy()#input
        results_df=pd.DataFrame()#output


    # ---model---
    print(f"loading model...\n")
    device=0 if torch.cuda.is_available() else -1
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=device
    )

    ## zsc
    results= []
    start_time = time.time()
    if by_lang:
        print(f"zsc by lang !")

    for idx, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="ZSC text..."):
        result=zsc_text(row, classifier, by_lang, dict_items)
        results.append(result)

        # ---save every----
        if (idx + 1) % save_interval == 0:
            results_df = pd.concat([results_df, pd.DataFrame(results)], ignore_index=True)
            results_df.to_csv(path_results, index=False)
            results=[]# 重置
            # results_df不断增加并保存，results储存每save_interval条结果
            print(f"[INTERVAL SAVE] {len(results_df)} / {len(df_unique)} rows zsced and saved!\n")

    # ---final save---
    if results:
        results_df = pd.concat([results_df, pd.DataFrame(results)], ignore_index=True)
        results_df.to_csv(path_results, index=False)
        print(f"✅ [FINAL SAVE] {len(results_df)}/{len(df_unique)} rows saved to {path_results}!\n")

    end_time = time.time()
    print(f"\n[DONE] ZSC sur {len(df)} textes avec {len(dict_items['en'])} EN labels/{len(dict_items['fr'])} FR labels \n"
          f"par {model_name} prend {(end_time - start_time)/3600:.2f} hours ( {(end_time - start_time):.2f} sec)!\n")


    # --merge--
    cols_to_merge=['host_about']+labels_en
    results_to_merge=results_df[cols_to_merge]
    df_zsc=df.merge(results_to_merge, left_on="host_about", right_on='host_about', how='left')
    print(f"df_zsc: {df_zsc.shape};")


    return df_zsc


In [ ]:
for i,paths in input_output.items():
  print("\n",f"{i}".center(100,"="))
  print(f"[load]{paths['listings']}")
  df_input=pd.read_csv(paths['listings'])

  df_zsc=run_zsc(
      df_input,
      model_name="MoritzLaurer/bge-m3-zeroshot-v2.0",
      by_lang=True,
      path_results=paths['results_zsc'],
      save_interval=1000,
      )
  df_zsc.to_csv(paths['listings_zsc'], index=False)
  print(f"[save]{len(df_zsc)} df_zsc saved to {paths['listings_zsc']}\n")



 =============================================paris2406==============================================
[info] df: 62738; df_unique : 16206

[info] 18449 rows already zsced; 0 rows to zsc!

loading model...



Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

zsc by lang !


ZSC text...: 0it [00:00, ?it/s]


[DONE] ZSC sur 62738 textes avec 22 EN labels/22 FR labels 
par MoritzLaurer/bge-m3-zeroshot-v2.0 prend 0.00 hours ( 0.01 sec)!

df_zsc: (62738, 107);


...

[INTERVAL SAVE] 9268 / 9620 rows zsced and saved!



ZSC text...:  98%|█████████▊| 9460/9620 [1:03:07<01:51,  1.44it/s]

[INTERVAL SAVE] 9460 / 9620 rows zsced and saved!



ZSC text...: 100%|██████████| 9620/9620 [1:04:10<00:00,  2.50it/s]


✅ [FINAL SAVE] 9620/9620 rows saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris2306/resultsZSC_paris2306.csv!


[DONE] ZSC sur 32901 textes avec 22 EN labels/22 FR labels 
par MoritzLaurer/bge-m3-zeroshot-v2.0 prend 1.07 hours ( 3850.55 sec)!

df_zsc: (32901, 108);
[save]32901 df_zsc saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris2306/listings_zsc_paris2306.csv



In [14]:
groups=['paris_2312',"london_2312"]
for i in groups:
  paths=input_output[i]
  print("\n",f"{i}".center(100,"="))
  print(f"[load]{paths['listings']}")
  df_input=pd.read_csv(paths['listings'])

  df_zsc=run_zsc(
      df_input,
      model_name="MoritzLaurer/bge-m3-zeroshot-v2.0",
      by_lang=True,
      path_results=paths['results_zsc'],
      save_interval=1000,
      )
  df_zsc.to_csv(paths['listings_zsc'], index=False)
  print(f"[save]{len(df_zsc)} df_zsc saved to {paths['listings_zsc']}\n")
  # ~10k 1h


 =============================================paris_2312=============================================
[load]/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2312/listings_processed_paris_2312.csv
[info] df: 35829; df_unique : 9794

[info] 19988 rows already zsced; 445 rows to zsc!

loading model...



Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

zsc by lang !


ZSC text...: 100%|██████████| 445/445 [02:40<00:00,  2.77it/s]


✅ [FINAL SAVE] 20433/9794 rows saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2312/resultsZSC_paris_2312.csv!


[DONE] ZSC sur 35829 textes avec 22 EN labels/22 FR labels 
par MoritzLaurer/bge-m3-zeroshot-v2.0 prend 0.04 hours ( 161.76 sec)!

df_zsc: (35829, 114);
[save]35829 df_zsc saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/paris_2312/listings_zsc_paris_2312.csv


 ============================================london_2312=============================================
[load]/content/Drive/MyDrive/airbnbMM2/host_about/data_processed/london_2312/listings_processed_london_2312.csv
[info] df: 43172; df_unique : 11141

loading model...



...

[INTERVAL SAVE] 9777 / 11141 rows zsced and saved!



ZSC text...: 100%|█████████▉| 11119/11141 [1:08:06<00:11,  1.89it/s]

[INTERVAL SAVE] 11119 / 11141 rows zsced and saved!



ZSC text...: 100%|██████████| 11141/11141 [1:08:14<00:00,  2.72it/s]


✅ [FINAL SAVE] 11141/11141 rows saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/london_2312/resultsZSC_london_2312.csv!


[DONE] ZSC sur 43172 textes avec 22 EN labels/22 FR labels 
par MoritzLaurer/bge-m3-zeroshot-v2.0 prend 1.14 hours ( 4094.88 sec)!

df_zsc: (43172, 112);
[save]43172 df_zsc saved to /content/Drive/MyDrive/airbnbMM2/host_about/data_processed/london_2312/listings_zsc_london_2312.csv

